In [ ]:
import json
import sys
import os
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import KFold
import torch
from torch.utils.data import DataLoader
from collections import defaultdict
from typing import Dict, Any, Tuple, Optional, Callable


models_path = os.path.abspath(os.path.join('..', 'models'))
sys.path.append(models_path)

models_path = os.path.abspath(os.path.join('..', 'src'))
sys.path.append(models_path)

from collate_batch import collate_batch
from answer_set import AnswerSet
from DKT.dkt_evaluation import calculate_loss
from DKT.dkt_model import DKT
from DKT.dkt_process import process


In [21]:
df_answers = pd.read_csv('../data/preprocessed/answers_df.csv')

num_problems = df_answers['problem_id'].nunique()
df_answers['problem_id_reindexed'] = df_answers['problem_id'].rank(method='dense').astype(int)
print(f"Number of unique problems: {num_problems}")

dict_answers = defaultdict(list)

# Group by 'user_id' and iterate through each group
for user_id, user_group in df_answers.groupby('user_id'):
    dict_answers[user_id] = list(zip(user_group['problem_id_reindexed'], user_group['correct']))

output_file = '../data/preprocessed/answers_dict_reindexed.json'
# Save the user dictionary to a JSON file
with open(output_file, 'w') as json_file:
    json.dump(dict_answers, json_file)
print(f"Reindexed user dictionary saved to {output_file}.")


Number of unique problems: 8266
Reindexed user dictionary saved to ../data/preprocessed/answers_dict_reindexed.json.


The number of unique exercises is too high, we need random vector representations.
Based on the paper we should transform to the following dimension:

In [22]:
embed_dim = round(np.log(num_problems))
print(embed_dim)

9


# Set constants

In [23]:
NUM_EPOCHS = 15
BATCH_SIZE = 100
NUM_FOLDS = 5  # Number of folds for cross-validation

train_ratio = 0.8  # 80% for training, 20% for testing

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Set configs

In [24]:
configs = [
    {
        "learning_rate": lr,
        "models_params": {
            "num_items": num_problems,
            "embed_dim": embed_dim,
            "hid_size": 200,
            "num_hid_layers": num_hid_layers,
            "drop_prob": drop_prob,
        },
    }
    for lr in [1e-3, 1e-4, 1e-5]  # 3 reasonable options for learning rate
    for num_hid_layers in [1, 2]  # Hidden layers 1 or 2
    for drop_prob in [0.3, 0.4, 0.5]  # Dropout rate 0.3, 0.4, 0.5
]

# Example: Printing configurations
for idx, config in enumerate(configs, 1):
    print(f"Config {idx}:\n{config}\n")



Config 1:
{'learning_rate': 0.001, 'models_params': {'num_items': 8266, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 1, 'drop_prob': 0.3}}

Config 2:
{'learning_rate': 0.001, 'models_params': {'num_items': 8266, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 1, 'drop_prob': 0.4}}

Config 3:
{'learning_rate': 0.001, 'models_params': {'num_items': 8266, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 1, 'drop_prob': 0.5}}

Config 4:
{'learning_rate': 0.001, 'models_params': {'num_items': 8266, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 2, 'drop_prob': 0.3}}

Config 5:
{'learning_rate': 0.001, 'models_params': {'num_items': 8266, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 2, 'drop_prob': 0.4}}

Config 6:
{'learning_rate': 0.001, 'models_params': {'num_items': 8266, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 2, 'drop_prob': 0.5}}

Config 7:
{'learning_rate': 0.0001, 'models_params': {'num_items': 8266, 'embed_dim': 9, 'hid_size': 200, 'num_hid_layers': 

# Split data into train and test sets

In [25]:

# Get all keys and shuffle them
keys = list(dict_answers.keys())
random.shuffle(keys)

# Split keys into train and test
split_index = int(len(keys) * train_ratio)
train_keys = keys[:split_index]
test_keys = keys[split_index:]

# Create train and test dictionaries
train_dict = {key: dict_answers[key] for key in train_keys}
test_dict = {key: dict_answers[key] for key in test_keys}


In [26]:
def train_dkt(
    model_params: Dict[str, Any],
    lr: float,
    num_epochs: int,
    device: torch.device,
    train_loader: DataLoader,
    val_loader: Optional[DataLoader] = None
) -> Tuple[torch.nn.Module, float]:
    """
    Trains a Deep Knowledge Tracing (DKT) model using the provided parameters.

    Args:
        model_params (dict): A dictionary of model parameters to initialize the DKT model.
        lr (float): Learning rate for the optimizer.
        num_epochs (int): Number of epochs to train the model.
        device (torch.device): The device (e.g., 'cuda' or 'cpu') to run the training on.
        train_loader (DataLoader): DataLoader for the training dataset.
        val_loader (Optional[DataLoader]): DataLoader for the validation dataset. If None, 
                                           training data is used for evaluation.

    Returns:
        Tuple[torch.nn.Module, float]: The trained DKT model and the best validation AUC achieved.

    Notes:
        - The function stops early if the validation AUC does not improve for 3 consecutive epochs.
        - If `val_loader` is not provided, the training data is used for evaluation, which may 
          lead to overestimation of the performance.
    """
    # Initialize the model and move it to the specified device
    model = DKT(**model_params).to(device)
    
    # Initialize the optimizer with the model's parameters and specified learning rate
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Variables to track the best validation AUC and early stopping condition
    best_auc = 0.0
    consecutive_epochs_no_improve = 0

    # Training loop for the specified number of epochs
    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}\n")

        # Training phase: Update the model using the training data
        process(model, train_loader, calculate_loss, device, optimizer)

        # Validation phase: Evaluate the model on the validation dataset (if provided)
        if val_loader is not None:
            _, val_auc = process(model, val_loader, device, calculate_loss)
        else:
            # Use training data for evaluation if no validation DataLoader is provided
            print("No validation loader provided. Using training data for evaluation.")
            _, val_auc = process(model, train_loader, device, calculate_loss)
        
        # Print the validation AUC for the current epoch
        print(f"Validation AUC: {val_auc:.4f}, Best AUC so far: {best_auc:.4f}")

        # Check if the validation AUC has improved
        if val_auc > best_auc:
            best_auc = val_auc
            consecutive_epochs_no_improve = 0  # Reset counter if there is an improvement
        else:
            consecutive_epochs_no_improve += 1  # Increment counter if no improvement

        # Stop training early if no improvement in validation AUC for 3 consecutive epochs
        if consecutive_epochs_no_improve == 3:
            print("AUC did not improve for 3 consecutive epochs. Training stopped early.")
            break

    # Return the trained model and the best validation AUC achieved
    return model, best_auc


In [27]:
def k_fold_cv_dkt(
    num_folds,
    model_params: Dict[str, Any],
    lr: float,
    num_epochs: int,
    device: torch.device,
    train_dict: Dict[int, Any],
    batch_size: int = 100,
    collate_batch: Callable = None,
    num_workers: int = 2
) -> float:
    """
    Perform k-fold cross-validation for the Deep Knowledge Tracing (DKT) model.

    Args:
        num_folds (int): Number of folds for cross-validation.
        model_params (Dict[str, Any]): Parameters for initializing the DKT model.
        lr (float): Learning rate for the optimizer.
        num_epochs (int): Number of epochs for training in each fold.
        device (torch.device): The device (e.g., 'cuda' or 'cpu') to run the training on.
        train_dict (Dict[int, Any]): Dictionary mapping keys to data for training.
        collate_batch (callable, optional): Function to merge a list of samples into a batch. Defaults to None.
        batch_size (int, optional): Batch size for training and validation. Defaults to 32.
        num_workers (int, optional): Number of subprocesses to use for data loading. Defaults to 2.

    Returns:
        float: The average validation AUC across all folds.

    Notes:
        - Assumes that the `train_dict` keys can be split into train and validation sets.
        - Requires an `AnswerSet` dataset and the `train_dkt` training function.
    """
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    
    train_keys = list(train_dict.keys())

    val_auc_list = []

    for fold, (fold_train_idx, fold_val_idx) in enumerate(kf.split(train_keys)):
        print(f"\nFold {fold+1}/{num_folds}")
        fold_train_keys = [train_keys[i] for i in fold_train_idx]
        fold_val_keys = [train_keys[i] for i in fold_val_idx]

        fold_train_dict = {key: train_dict[key] for key in fold_train_keys}
        fold_val_dict = {key: train_dict[key] for key in fold_val_keys}

        fold_train_dataset = AnswerSet(fold_train_dict)
        fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, collate_fn=collate_batch, pin_memory=True, num_workers=num_workers)

        fold_val_dataset = AnswerSet(fold_val_dict)
        fold_val_loader = DataLoader(fold_val_dataset, batch_size=batch_size, collate_fn=collate_batch, pin_memory=True, num_workers=num_workers)

        _, val_auc = train_dkt(
            model_params, lr, num_epochs, device,
            fold_train_loader, fold_val_loader
            )

        val_auc_list.append(val_auc)

    val_auc_avg = sum(val_auc_list) / num_folds

    return val_auc_avg


In [ ]:
print('a')

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
(fold_train_idx, fold_val_idx) = kf.split(train_keys)

# Hyperparameter-tuning

In [ ]:
best_val_auc_avg = 0  # Track the best validation AUC
best_config = None  # Track the best configuration

for idx, config in enumerate(configs, 1):
    print(f"\nEvaluating Config {idx}/{len(configs)}")

    lr = config['learning_rate']
    model_params = config['models_params']

    val_auc_avg = k_fold_cv_dkt(
        num_folds=NUM_FOLDS,
        model_params=model_params,
        lr=lr,
        num_epochs=NUM_EPOCHS,
        device=device,
        train_dict=train_dict,
        batch_size=BATCH_SIZE,
        collate_batch=collate_batch,
        )

    # Update the global best if needed
    if val_auc_avg > best_val_auc_avg:
        best_val_auc_avg = val_auc_avg
        best_config = config  # Save the best configuration
        print(f"\nNew best model found: Config {idx}. Validation AUC: {best_val_auc_avg:.4f}")

print(f"\nBest test AUC: {best_val_auc_avg:.4f}")
print(f"\nBest Configuration: {best_config}")



Evaluating Config 1/18

Fold 1/5

Epoch 1

